In [75]:
#Start

import pandas as pd, torch
from PIL import Image
import torch
import cv2
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import numpy as np
import os 
from sklearn.model_selection import train_test_split 
import torch.nn as nn  
import matplotlib.pyplot as plt
import warnings

class CustomImageDataset(Dataset):
    def __init__(self, label_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(label_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform
        self.dictionary = {"Tett bebyggelse":0,"Spredt bebyggelse":1,"Jordbruk":2,"Skog":3,"Fjell / snaufjell":4,"Myr / våtmark":5,"Innsjø / elv":6,"Kyst / hav":7,"Bre / snø":8}


    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        label = self.img_labels.iloc[idx, 1]
        fixed_label = label.replace(" / ", "_")
        more_fixed_label = fixed_label.replace(" ", "_")
        removed_ø = more_fixed_label.replace("ø","o")
        removed_å = removed_ø.replace("å","a")
        label_path = os.path.join(self.img_dir, removed_å)
        img_path = os.path.join(label_path,self.img_labels.iloc[idx,0])
        image = cv2.imread(img_path)
        transform = transforms.Compose([
            transforms.ToTensor()
        ])
        tensor = transform(image)
        label_number = self.dictionary[label]
        if self.transform:
            tensor = self.transform(tensor)
        if self.target_transform:
            label = self.target_transform(label)
        return tensor, label_number

dataset = CustomImageDataset(label_file="kartdatasett_1169\kartdatasett\etiketter.csv", img_dir="kartdatasett_1169\kartdatasett")

train_dataloader = DataLoader(dataset, batch_size=3, shuffle=True)
dataset



<>:51: SyntaxWarning: invalid escape sequence '\k'
<>:51: SyntaxWarning: invalid escape sequence '\k'
<>:51: SyntaxWarning: invalid escape sequence '\k'
<>:51: SyntaxWarning: invalid escape sequence '\k'
C:\Users\felix\AppData\Local\Temp\ipykernel_8004\1319728614.py:51: SyntaxWarning: invalid escape sequence '\k'
  dataset = CustomImageDataset(label_file="kartdatasett_1169\kartdatasett\etiketter.csv", img_dir="kartdatasett_1169\kartdatasett")
C:\Users\felix\AppData\Local\Temp\ipykernel_8004\1319728614.py:51: SyntaxWarning: invalid escape sequence '\k'
  dataset = CustomImageDataset(label_file="kartdatasett_1169\kartdatasett\etiketter.csv", img_dir="kartdatasett_1169\kartdatasett")


In [71]:
class ConvModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3,   
                        out_channels=1,   
                        kernel_size=3,   
                        stride=1)
        self.conv2 = nn.Conv2d(in_channels=1,
                      out_channels=1,
                      kernel_size=5,
                      stride=1)
        self.conv3 = nn.Conv2d(in_channels=1,
                      out_channels=1,
                      kernel_size=6,
                      stride=2)
        self.relu = nn.ReLU()
        self.linear = nn.Linear(251,9)
        self.softmax = nn.Softmax(dim=0)

    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.relu(x)
        x = self.linear(x)
        x = self.softmax(x)
        return x

model = ConvModel()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

def run_one_epoch():
    running_loss = 0
    last_loss = 0
    for i, data in enumerate(train_dataloader):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        print(labels.shape)
        print(outputs.shape)
        loss = loss_fn(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss = loss.item()
        if i % 300 == 299:
            last_loss = running_loss/300
            print(f"loss {last_loss}")
            running_loss = 0
    return last_loss

for epochs in range(12):
    model.train(True)
    avg_loss = run_one_epoch()
    model.eval()
        
        
        

conv_train1 = conv_filter(X_train)
print(conv_train1)


torch.Size([3])
torch.Size([3, 1, 251, 9])


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of dimension: 1